# 10-数字接口与模电应用基础


## 1. 本篇定位与资料边界

本篇是对 01—09 的延伸补充，主题是把电压、电流、三极管、MOS 管、继电器、NE555、测量和 PCB 知识接到嵌入式常见接口。

原课程分享中的视频到 `64_电子琴的pcb布局.mp4` 结束，没有视频 65 之后的原始内容。因此本篇不是课程第 10 节视频的转录，而是为“模电基础 → 嵌入式硬件接口”准备的专题笔记。

覆盖内容：

- GPIO 的输入、推挽输出、上拉、下拉和开漏。
- 按键消抖、LED 限流、三极管/MOS 管驱动和继电器保护。
- ADC 采样、分压、量化、参考电压和输入阻抗。
- PWM 调光、调速、RC 滤波和输出负载。
- UART、I²C、SPI 的电气特点，以及电源、接地、去耦、保护和调试。

本文使用平台无关的符号和公式。具体 MCU 的电平阈值、GPIO 最大电流、ADC 参考电压、采样时间、复用功能和绝对最大额定值必须以所用型号的数据手册为准。


## 2. 模拟量与数字量：先问“连续还是离散”

模拟信号在一定范围内连续变化，例如传感器电压、麦克风输出和电池电压。数字信号把电压解释为有限的逻辑状态，例如 0 和 1。

数字输入并不是“低于某个电压就绝对是 0、高于某个电压就绝对是 1”这么简单。器件会给出输入低电平最大值 `VIL(max)` 和输入高电平最小值 `VIH(min)`；输出也会给出 `VOL(max)` 与 `VOH(min)`。

| 区域 | 含义 | 设计处理 |
| --- | --- | --- |
| $V\le V_{IL(max)}$ | 接收端保证识别为低 | 保持足够余量 |
| $V\ge V_{IH(min)}$ | 接收端保证识别为高 | 保持足够余量 |
| 两者之间 | 可能不确定 | 不要把它当作稳定逻辑 |

两个器件直接连接时，检查噪声裕量：

$$NM_H=V_{OH(min)}-V_{IH(min)}$$

$$NM_L=V_{IL(max)}-V_{OL(max)}$$

噪声裕量越小，电源波动、地电位差、串扰和长线反射越容易造成误判。数字电路仍然受模拟电压、电流、时间和阻抗支配。


## 3. GPIO 输入：浮空、上拉和下拉

GPIO 设为输入时，输入端通常只需要很小的漏电流，但不能让它长期浮空。浮空节点容易拾取环境噪声，在逻辑 0 和 1 之间随机变化。

三种常用状态：

| 输入方式 | 默认状态 | 常见用途 | 关键检查 |
| --- | --- | --- | --- |
| 外部下拉 | 松开/断开时为 0 | 按键按下接 VCC | 下拉阻值、按下电流 |
| 外部上拉 | 松开/断开时为 1 | 按键按下接 GND | 上拉阻值、低电平能力 |
| 内部上拉/下拉 | 由 MCU 内部电阻提供 | 简单按键、开漏总线 | 阻值范围和是否真的启用 |

上拉电阻的选择要同时满足两点：

1. 低电平时流过的电流不能让开关或输入端超限。
2. 高电平时要能抵抗漏电、噪声和外部电容造成的下降。

若上拉为 `R_P`、供电为 `VCC`，按键按下接地时的电流近似为：

$$I_{key}\approx\frac{V_{CC}}{R_P}$$

阻值越小，抗干扰和上升速度通常更好，但静态功耗更大；阻值越大，功耗更小，但节点更容易受噪声和电容影响。


## 4. GPIO 输出：推挽与开漏

### 推挽输出

推挽输出可以主动拉高，也可以主动拉低，适合驱动单一接收端或普通控制线。输出高电平时，端口向负载供电；输出低电平时，端口吸收负载电流。

不能把两个推挽输出直接相连后让它们输出相反电平，否则会形成近似短路。多个设备共享一条线时，应改用规定的总线结构。

### 开漏/开集电极输出

开漏输出通常只主动拉低，释放时由上拉电阻把总线拉高：

```text
VCC ── R_P ── 总线 ── 多个开漏输出
                         └─ 只能拉低或释放
```

I²C、告警线、线与逻辑等常使用这种方式。多个设备同时释放不会冲突；只要有一个设备拉低，总线就是低电平。

开漏总线的高电平上升速度由上拉电阻和总线电容共同决定：

$$t\text{ 的上升趋势}\sim R_P C_{bus}$$

上拉太大，上升慢且抗干扰弱；上拉太小，低电平时电流过大。实际值要结合总线速率、设备数量、电容和数据手册。


## 5. 按键输入：机械触点为什么需要消抖

机械按键闭合或断开时，触点会在几毫秒内反复弹跳。MCU 采样得足够快，就可能把一次按压读成多次边沿。

### 硬件消抖

常见方法是按键配合 RC 网络和施密特触发输入。若串联电阻为 `R`、电容为 `C`，时间常数为：

$$\tau=RC$$

RC 只能让边沿变慢，最终还需要具有明确阈值和滞回的输入，避免信号长时间停留在不确定区域。`R`、`C` 的值应结合输入漏电、采样速度和功耗选择。

### 软件消抖

一种通用流程：

1. 定时采样输入。
2. 只有连续若干次读到相同状态，才更新稳定状态。
3. 稳定状态从松开变为按下时，只产生一次事件。
4. 释放时同样确认稳定，防止长按重复触发。

10—20 ms 可以作为低速人机按键的起始试验范围，但不是所有产品的固定答案。快速按键、旋钮、触摸输入和低功耗唤醒需要重新设计。


## 6. LED 与 GPIO：用电阻定义电流

LED 不是可以直接接到 GPIO 的“普通灯泡”。它的正向压降随颜色、电流和温度变化；没有限流时，电流可能由端口内阻、LED 和电源共同决定，容易超出器件额定值。

低端 LED 的初步计算：

$$R_{LED}\approx\frac{V_{OH}-V_F-V_{SW}}{I_{LED}}$$

例：假设 GPIO 高电平 3.3 V、LED 正向压降 2.0 V、串联开关压降 0.1 V、目标电流 5 mA：

$$R\approx\frac{3.3-2.0-0.1}{0.005}=240\,\Omega$$

可选 270 Ω，再估算实际电流：

$$I\approx\frac{1.2}{270}\approx4.4\text{ mA}$$

电阻功率约为：

$$P_R\approx I^2R\approx(4.4\text{ mA})^2\times270\approx5.2\text{ mW}$$

还要核对 MCU 每个 GPIO 的源电流、灌电流、端口总电流和 LED 峰值电流。PWM 降低平均亮度时，瞬时电流仍不能超过脉冲额定值。


## 7. 用三极管或 MOS 管驱动负载

GPIO 适合发送控制信号，不能默认直接承担继电器、马达、较大灯带或喇叭的功率。驱动级把“控制电流”和“负载电流”分开。

### N 沟道 MOS 管低端驱动

```text
VLOAD → 负载 → D  MOSFET  S → GND
                 G ← GPIO（串小电阻或按手册设计）
```

选型要看：

- 目标栅源电压下的 $R_{DS(on)}$，不是只看 $V_{GS(th)}$。
- 负载持续电流、启动电流和脉冲电流。
- 导通损耗：

$$P_{MOS}\approx I_D^2R_{DS(on)}$$

- 栅极总电荷、开关频率、封装散热和关断状态。

### 三极管低端驱动

三极管开关通常需要基极电阻。以强迫电流放大系数估算：

$$I_B\ge\frac{I_C}{\beta_{forced}}$$

$$R_B\approx\frac{V_{GPIO}-V_{BE}}{I_B}$$

这是初选方法；应以具体器件在饱和条件下的测试数据为准，并确认 GPIO 仍能保持所需高电平。

控制信号进入驱动器件前后，分别检查 GPIO 电平、栅极/基极电压、负载两端电压和器件温升。


## 8. 继电器、马达和线圈：关断时会发生什么

电感电流不能瞬间变为零。继电器线圈、直流马达、电磁阀和电磁铁关断时，会通过寄生电容和器件击穿寻找电流路径，产生反向高压。

低侧驱动的线圈常并联续流二极管：

```text
VCOIL ── 线圈 ── 开关 ── GND
   └──── 二极管 ─────────┘
```

二极管正常通电时应反向截止；开关关断、线圈两端电压反向时，二极管为电流提供受控路径。二极管的反向耐压、正向电流和脉冲能力要与线圈匹配。

马达需要更快关断或更强抗干扰时，可能使用 TVS、RC 吸收、齐纳钳位或专用驱动器。保护器件会改变关断速度，不能只凭“并一只二极管”判断所有应用都完成。

继电器触点侧还要单独检查 COM、NO、NC、负载类型、浪涌、电压等级、爬电距离和隔离边界。低压线圈控制安全，不代表触点侧可以随意接触。


## 9. ADC：把电压变成数字码

理想 N 位 ADC 使用参考电压 `VREF`，把输入范围分成有限数量的码。理想分辨率近似为：

$$LSB\approx\frac{V_{REF}}{2^N}$$

常见的理想量化码可写为：

$$Code\approx\operatorname{round}\left(\frac{V_{IN}}{V_{REF}}(2^N-1)\right)$$

例如 12 位 ADC、`VREF=3.3 V`、输入 1.65 V：

$$Code\approx\frac{1.65}{3.3}\times4095\approx2048$$

这只是理想估算。实际误差还来自：

- 参考电压误差和温漂。
- ADC 偏置、增益、积分非线性和微分非线性。
- 输入源阻抗、采样电容和采样时间。
- 电源噪声、地弹噪声、布局串扰和量化噪声。

ADC 读数应记录位数、参考电压、通道、采样时间、平均方式和校准条件。只写“读到 2048”无法复现测量。


## 10. ADC 分压：把被测范围压进输入范围

若被测电压可能超过 ADC 输入范围，可用分压器：

```text
VIN ── R1 ── ADC_IN ── R2 ── GND
```

理想空载输出：

$$V_{ADC}=V_{IN}\frac{R_2}{R_1+R_2}$$

例：测量最高 12 V、ADC 允许最高 3.3 V，取 `R1=27 kΩ`、`R2=10 kΩ`：

$$V_{ADC,max}=12\times\frac{10}{27+10}\approx3.24\text{ V}$$

分压静态电流约为：

$$I_{div}\approx\frac{12}{37\text{ kΩ}}\approx0.324\text{ mA}$$

设计时留出容差和瞬态余量，不能把 3.3 V 作为毫无误差的硬边界。ADC 输入采样会使分压器看到额外负载；必要时降低阻值、延长采样时间或用运放缓冲。

若 ADC 节点加电容，分压电阻与电容组成低通网络。它能降低噪声，也会减慢响应；采样周期、稳定时间和滤波参数需要一起选择。


## 11. 传感器接入：源阻抗、滤波与校准

传感器接口先分为三类：

1. **电压输出型：**直接送 ADC，但要检查输出范围、源阻抗和过压。
2. **电阻变化型：**与固定电阻组成分压，再测量节点电压。
3. **电流输出型：**需要采样电阻或跨阻放大器，把电流变为 ADC 可测电压。

源阻抗过高时，ADC 内部采样电容在采样时间内可能充不满，导致读数偏低或随采样顺序变化。可用以下方法改善：

- 降低分压电阻值，同时接受更高静态功耗。
- 增加 ADC 采样时间。
- 在 ADC 端增加电容并等待稳定。
- 使用输入偏置和噪声符合要求的缓冲运放。

校准分为两层：

- **电气校准：**用已知电压校正零点和比例。
- **物理校准：**把电压映射为温度、压力、光照等工程量。

不要把“ADC 码稳定”当作“传感器准确”。稳定性、重复性、线性度和绝对误差需要分别记录。


## 12. PWM：用占空比控制平均效果

PWM 在固定周期内快速切换高低电平，用占空比表达平均作用：

$$D=\frac{t_{ON}}{T}\times100\%$$

理想低损耗负载下，平均电压可粗略理解为：

$$V_{avg}\approx D\times V_{HIGH}$$

但 LED 的亮度、马达的转矩和喇叭的声音并不只由平均电压决定，还受到频率、器件响应、负载惯性、驱动器和人耳特性的影响。

定时器频率常近似为：

$$f_{PWM}=\frac{f_{timer}}{(PSC+1)(ARR+1)}$$

占空比近似为：

$$D\approx\frac{CCR}{ARR+1}$$

不同 MCU 对寄存器边界和计数模式的定义可能不同，正式代码应查参考手册。

选择频率时要同时考虑：可见闪烁、马达啸叫、开关损耗、分辨率和 EMI。频率升高并不总是更好。


## 13. PWM 负载：LED、马达和蜂鸣器的区别

| 负载 | PWM 主要效果 | 需要重点检查 |
| --- | --- | --- |
| LED | 平均光照变化 | 峰值电流、限流、频率、热和视觉闪烁 |
| 直流马达 | 平均转矩与速度变化 | 启动电流、续流、噪声、MOS 管温升 |
| 无源蜂鸣器 | 频率决定音高，占空比影响音色 | 频率范围、驱动电流、耦合方式 |
| 有源蜂鸣器 | 触发后内部自行发声 | 不能把它当任意音高负载 |

PWM 输出端仍是数字开关。负载较大时要使用三极管、MOS 管或专用驱动器，不要只修改占空比就超出 GPIO 电流能力。

LED PWM 调光时，峰值电流、占空比和平均功耗都需要受控。马达 PWM 的关断回路必须有续流或其他钳位。蜂鸣器换成喇叭后，阻抗、功率和交流耦合条件都会改变。


## 14. RC 低通：把 PWM 变成近似模拟电压

PWM 后接电阻和电容可以得到平均值附近的电压：

```text
PWM ── R ── VOUT
             │
             C
             │
            GND
```

截止频率近似为：

$$f_c=\frac{1}{2\pi RC}$$

要降低 PWM 纹波，通常让 `f_PWM` 明显高于 `f_c`；但 `f_c` 越低，输出响应越慢。若后级负载不够高阻，负载会改变 R、C 的等效关系，必要时增加缓冲器。

RC 滤波器不是高精度 DAC：

- 需要等待稳定，不能立刻得到新电压。
- PWM 频率和占空比抖动会转化为输出纹波。
- 电阻、电容、负载和 ADC/运放输入都会造成误差。
- 负载需要电流时，直接从滤波节点取电可能使电压塌陷。

验收时用示波器同时观察 PWM 节点和滤波后节点，记录纹波峰峰值、响应时间和负载条件。


## 15. UART、I²C、SPI：先看电气结构，再看协议

### UART

UART 常用独立 TX、RX 和共享 GND。双方电平、波特率、数据位、校验和停止位要匹配；TX 与 RX 交叉连接。UART 不是天然的 5 V/3.3 V 电平转换器，供电不同的设备要先确认电平兼容。

### I²C

I²C 的 SDA、SCL 通常是开漏/开集电极结构，需要上拉电阻。上拉值取决于总线电容、速率、供电和器件低电平能力。多个设备共享总线时检查地址冲突、上拉重复和最长线长。

### SPI

SPI 通常使用 SCK、MOSI、MISO 和每个从设备的 CS。它多用推挽驱动，速度高、协议简单，但线长、回流、片选时序和不同设备的模式参数要分别检查。

三种接口的共同检查：

- 信号参考地是否共用且电位差可接受。
- 高低电平是否符合输入阈值。
- 线缆长度、电容、边沿速度和终端是否匹配。
- 空闲状态、上电状态和复位状态是否会误触发负载。


## 16. 电平转换：3.3 V 与 5 V 不能只看“能不能亮”

5 V 输出接入只耐 3.3 V 的输入，可能超过绝对最大额定值；3.3 V 输出接入 5 V 输入，也不一定满足对方的 `VIH`。

选择方法：

| 场景 | 常见方案 | 适用边界 |
| --- | --- | --- |
| 慢速、单向、输入阻抗高 | 电阻分压 | 需计算输入电流、上升时间和容差 |
| 开漏双向总线 | 专用 I²C 电平转换器或合适 MOS 管结构 | 上拉两侧供电与速率要匹配 |
| 高速推挽单向 | 专用逻辑电平转换器 | 查看传播延迟、方向和边沿 |
| 低速控制线 | 三级管/MOS 管或开关器件 | 逻辑反相、漏电和速度要确认 |

分压器只能降低电压，不能把 3.3 V 自动升成可靠的 5 V 高电平。电平转换前先查看两侧数据手册的输入阈值、容限、上电状态和是否允许 5 V 容忍。


## 17. 电源去耦、接地与回流

去耦电容为芯片附近的瞬态电流提供低阻抗路径。常见做法是在每个数字 IC 的供电脚附近放置小容量陶瓷电容，并根据负载在电源入口或模块附近增加较大的储能电容。

布局重点：

- 小电容与供电脚、地脚形成的回路尽量短、宽。
- 负载大电流回流不要穿过 ADC 参考地或敏感传感器地。
- 功率路径、定时节点、晶体管基极和 ADC 输入分别考虑回流。
- 地平面不是“放上去就自动正确”，要检查回流是否被板框、分割和连接器绕远。
- 电源入口增加保险、反接保护、TVS 或限流时，按实际能量和故障模式选择。

用示波器看供电时，要同时记录探头接地方式。长鳄鱼夹可能把测量到的“噪声”变成测量回路自身的天线。


## 18. 输入保护与接口边界

外部接口可能遇到静电、插拔浪涌、反接、过压、短路和长线感应。保护方案要与信号速度、正常电压和故障能量匹配。

常见器件及作用：

| 器件 | 主要作用 | 不能替代什么 |
| --- | --- | --- |
| 串联电阻 | 限制瞬态电流、配合电容滤波 | 不能独自吸收大能量 |
| TVS | 钳位快速浪涌 | 不能把持续过压变成安全电压 |
| 肖特基钳位 | 把输入限制到电源轨附近 | 需要确认电源能吸收电流 |
| 保险丝/自恢复保险丝 | 限制持续过流 | 不能替代正确的电路设计 |
| 反接保护 MOS 管/二极管 | 限制反向供电 | 需核对压降和功耗 |
| RC 滤波 | 降低高频噪声 | 会降低边沿速度 |

外部接口的保护元件应靠近接口，受保护信号再进入 MCU。保护路径的回流不能把浪涌电流引入敏感模拟地。与用户可触及、长线或外部电源有关的接口，还要按实际产品标准和实验条件复核。


## 19. 一个综合练习：按键调光与电压监测

目标：用一个低压电源、一个按键、一个电位器或分压传感器和一颗 LED，练习 GPIO、ADC、PWM 与驱动分层。

### 功能框图

```text
按键 ── GPIO 输入/消抖 ──┐
                         ├─ MCU ── PWM ── MOS 管/限流 ── LED
电位器/传感器 ── 分压 ── ADC ┘
```

### 设计步骤

1. 先确定电源电压、LED 电流、ADC 输入范围和按键默认状态。
2. 按电源和 LED 压降计算限流电阻，核对 GPIO 或 MOS 管的驱动条件。
3. 让 ADC 分压的最高电压低于输入允许范围，并留出容差余量。
4. 按键使用上拉或下拉，软件连续采样消抖。
5. 用 ADC 读数映射到 PWM 占空比，限制上下限，避免异常输入得到 0% 或 100% 的危险状态。
6. 断电测短路，限流上电，先看电源和输入，再连接 LED 负载。
7. 记录 ADC 码、PWM 占空比、LED 电流、供电和温升。

### 验收条件

- 松开按键时输入稳定，不随机跳变。
- 电位器转动时 ADC 码单调变化，超量程有明确处理。
- LED 平均亮度随 PWM 改变，峰值电流和器件温升在范围内。
- 断开或短接传感器时，程序进入可预期的故障状态。
- 原理图、封装、PCB、BOM 和测试记录版本一致。

这是低压学习项目。没有实际搭建和测量时，结果应写“待验证”，不能把设计预期写成实测结论。


## 20. 统一调试流程：从电源到波形再到负载

遇到 GPIO、ADC、PWM 或外设异常时，按以下顺序缩小范围：

1. **确认供电：**测 MCU、传感器和负载的实际电压，观察上电瞬间是否下陷。
2. **确认参考地：**示波器、万用表和外部模块是否连接到同一个合理参考点。
3. **确认配置：**GPIO 方向、上下拉、复用功能、ADC 通道、PWM 频率和极性是否正确。
4. **确认输入：**用已知高低电平或已知电压代替传感器，排除外部模块。
5. **确认中间节点：**测按键节点、分压节点、MOS 栅极、三极管基极、定时电容和总线。
6. **确认输出：**先看 MCU 引脚，再看驱动器件，再看负载两端。
7. **一次只改一个变量：**记录参数、现象和测量条件。

| 现象 | 先测哪里 |
| --- | --- |
| GPIO 输入随机变化 | 输入脚、上拉/下拉、参考地、按键抖动 |
| LED 不亮 | GPIO 波形、限流电阻两端、LED 极性、驱动管栅极/基极 |
| MOS 管发热 | $V_{GS}$、$R_{DS(on)}$ 条件、负载电流、散热 |
| ADC 一直满量程 | 输入是否过压、分压比、参考电压、通道配置 |
| ADC 随采样顺序变化 | 源阻抗、采样时间、ADC 端电容、通道切换 |
| PWM 频率不对 | 定时器时钟、预分频、周期寄存器、计数模式 |
| UART 有乱码 | 电平、波特率、地、线序和数据格式 |
| I²C 无应答 | 上拉、地址、SDA/SCL 方向、复位和电平 |


## 21. 一页复习表

| 主题 | 关键结论 |
| --- | --- |
| 数字输入 | 检查 VIL、VIH 和噪声裕量，不能让输入浮空 |
| 推挽输出 | 能主动拉高和拉低；不能把相反的两个推挽输出直接相连 |
| 开漏输出 | 只能主动拉低，释放时靠上拉变高，适合共享总线 |
| 按键 | 机械触点会抖动；用 RC、施密特或软件连续采样消抖 |
| LED | 串联限流电阻；同时检查 GPIO 源/灌电流和端口总电流 |
| MOS 管 | 用目标 $V_{GS}$ 下的 $R_{DS(on)}$ 选型，阈值电压不是充分导通保证 |
| 线圈 | 关断会产生反向电压；配置续流或合适钳位，并检查触点侧边界 |
| ADC | $LSB\approx V_{REF}/2^N$；参考、采样时间和源阻抗影响精度 |
| 分压 | $V_{ADC}=V_{IN}R_2/(R_1+R_2)$；留容差和过压余量 |
| PWM | 占空比控制平均效果；频率、峰值电流和负载决定实际表现 |
| RC 滤波 | 降低纹波会增加响应时间，负载会改变滤波结果 |
| UART | TX/RX 交叉、共享地、电平和格式匹配 |
| I²C | SDA/SCL 通常开漏，需要上拉；地址和总线电容要核对 |
| SPI | SCK/MOSI/MISO/CS，关注模式、片选、回流和边沿 |
| 去耦 | 小电容靠近供电脚；大电流回流远离敏感模拟节点 |
| 保护 | 串阻、TVS、钳位、保险和反接保护按故障能量选，不可互相替代 |
| 调试 | 先电源和地，再输入和中间节点，最后驱动与负载 |


## 22. 自测题与参考答案

### 先独立作答

1. 为什么一个数字输入脚不应长期浮空？
2. 推挽输出和开漏输出的主要区别是什么？
3. 上拉电阻太大或太小各有什么代价？
4. 按键为什么需要消抖？10—20 ms 是否是所有按键的固定答案？
5. 3.3 V GPIO 驱动 LED 时，为什么仍要串联限流电阻？
6. 选择 MOS 管时，为什么不能只看 $V_{GS(th)}$？
7. 继电器线圈关断时为什么需要续流或钳位？
8. 12 位、3.3 V 参考的 ADC，1.65 V 输入的理想码约是多少？
9. ADC 分压最高输入接近 3.3 V 时，为什么还要留余量？
10. PWM 频率升高是否一定更好？
11. I²C 总线为什么需要上拉电阻？
12. DRC 通过后，为什么仍要进行低压上电和波形验证？

### 参考答案

1. 浮空节点容易受噪声、漏电和耦合影响，在高低电平之间随机变化。
2. 推挽能主动拉高和拉低；开漏通常只能主动拉低，释放后依靠上拉变高，适合共享线。
3. 太大时上升慢、抗干扰弱；太小时低电平静态电流大、器件负担重。
4. 机械触点会弹跳，使一次按压产生多次边沿。10—20 ms 只是低速人机按键的起始试验范围，需按实际速度和电路验证。
5. LED 正向压降不等于恒压源，没有限流时电流可能超过 LED 和 GPIO 的额定值。
6. 阈值只表示很小电流开始导通；要看目标 $V_{GS}$ 下的 $R_{DS(on)}$、电流、功耗和热条件。
7. 线圈电流不能瞬间消失，磁场塌缩会产生反向高压，可能损坏驱动器件或造成干扰。
8. 约 $1.65/3.3\times4095=2048$。
9. 参考、分压、电阻容差和瞬态都有误差；贴近上限可能导致过压或满量程失真。
10. 不一定。频率升高会降低每周期时间和某些纹波，但可能增加开关损耗、EMI、驱动压力或超出负载适用范围。
11. 开漏输出释放时需要上拉把总线拉到高电平；上拉值还要满足上升时间和低电平电流条件。
12. DRC 主要检查几何、间距和连通规则，不能保证电平、供电、引脚映射、采样精度、驱动能力和功能正确。


## 23. 与前面笔记及参考资料的对应关系

| 已有笔记 | 本篇延伸 |
| --- | --- |
| 01 电学与电路基础 | 电平、欧姆定律、功率和安全边界 |
| 02 常用元器件与安全 | 电阻、电容、电感、保险、接插件与蜂鸣器在接口中的作用 |
| 03 电路测量与仿真工具 | GPIO、ADC、PWM、总线和负载的测量方法 |
| 04 电路定律、功率与继电器 | 负载驱动、线圈续流、容量、功率和回流 |
| 05 三极管与 MOS 管 | GPIO 到驱动器件的接口设计 |
| 06 NE555 定时器与振荡电路 | PWM、RC 滤波、定时和频率的联系 |
| 07 PCB 设计与 EDA 入门 | 去耦、回流、接口保护、封装和布局 |
| 08 电笔与电子琴项目实战 | 感应输入、按键、音频负载和项目验收 |
| 09 模电基础综合复习与项目验收 | 统一的计算、测量、排错和版本检查 |

本篇的接口原则和公式是学习补充；没有指定某个 MCU，也没有声称已经在特定开发板上实测。正式设计请查：

- 所用 MCU 的数据手册和参考手册：GPIO 电平、最大电流、ADC、定时器、复用和绝对最大额定值。
- 所用传感器、继电器、马达、蜂鸣器、喇叭和 MOS 管的完整型号资料。
- 所用 EDA 的封装、设计规则、制造文件和装配要求。

原课程资料：[百度网盘课程分享](https://pan.baidu.com/s/1AZwoEpfx_2anpJ8p9ofD-Q?pwd=9988)，提取码 `9988`。

参考入口：[TI 半导体技术资料](https://www.ti.com/)、[嘉立创 EDA 官方文档](https://docs.lceda.cn/cn/)。具体电气参数以对应器件手册为准。
